# OncoRisk: Cancer Risk Assessment & Model Training Pipeline
This notebook documents the machine learning pipeline used by the **OncoRisk** platform.

### Pipeline Overview
1. **Datasets**: UCI Breast Cancer Wisconsin, UCI Cervical Cancer Risk Factors, Kaggle Survey Lung Cancer, HAM10000 Skin Lesion / ABCDE Criteria, Colorectal Screening Cohort, Prostate Risk Trial.
2. **Algorithm Comparison**: Logistic Regression, Random Forest, and XGBoost.
3. **Class Imbalance**: SMOTE (Synthetic Minority Over-sampling Technique) and balanced class weighting.
4. **Probability Calibration**: `CalibratedClassifierCV` via Platt Sigmoid scaling.
5. **Selection Metric**: Maximizing composite ROC-AUC and Recall (Sensitivity).
6. **Explainability**: SHAP / Normalized Feature Contribution decomposition.

In [1]:
import json
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_curve, auc

In [2]:
# Load trained model metadata summary
metadata_path = "../ml/saved_models/model_metadata.json"
with open(metadata_path, "r") as f:
    metadata = json.load(f)

for c_type, info in metadata.items():
    print(f"Cancer Type: {c_type.upper():12s} | Selected Model: {info['best_algorithm']:20s} | Dataset: {info['dataset_name']}")

### Model Performance Comparison Across Algorithms

In [3]:
comparison_rows = []
for c_type, data in metadata.items():
    for algo_name, metrics in data["algorithms_comparison"].items():
        comparison_rows.append({
            "Cancer Type": c_type.capitalize(),
            "Algorithm": algo_name,
            "ROC-AUC": metrics["roc_auc"],
            "Recall": metrics["recall"],
            "Precision": metrics["precision"],
            "F1-Score": metrics["f1"],
            "Accuracy": metrics["accuracy"]
        })

df_comparison = pd.DataFrame(comparison_rows)
print(df_comparison.to_string(index=False))

### Feature Importance & Attributions
Top risk factors decomposed for clinical transparency and patient empowerment.

In [4]:
for c_type in ["breast", "lung", "cervical", "skin", "colorectal", "prostate"]:
    print(f"\nTop 4 Contributing Factors for {c_type.upper()}:")
    factors = metadata[c_type]["feature_importances"][:4]
    for f in factors:
        print(f"  - {f['feature']:25s}: {f['weight']:.1f}%")